# 5단계: 추천 파이프라인 실험

4단계 e5-base 임베딩(텍스트 A/B)을 재사용해 자연어 입력에 대한 Top-K 추천을 실험한다.

```
입력 -> 조건 추출(preprocessing) -> 후보 검색(retrieval) -> 필수 필터, 선호 재랭킹, 중복 제어(ranking) -> Top-K(recommendation)
```

- 후보는 업체명이 없는 공공 데이터만 쓴다. 프랜차이즈 메뉴는 로드할 때 제외한다.
- 라벨은 모델 추정이라 조건 준수 지표는 저장된 라벨 기준이며 정확도가 아니다. 점수는 정렬용 값이다.
- 출력은 `data/processed/recommendation/`에 저장한다.

## 1. 모듈 로드 및 실행 환경

In [1]:
import json
import sys
import time
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src").is_dir() and (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Menu_recommend 저장소 안에서 실행해 주세요")
sys.path.insert(0, str(PROJECT_ROOT))

from src.embedding import DEFAULT_SPEC, E5Embedder, EmbeddingStore, describe_environment, source_hashes
from src.preprocessing import parse_query, support_table
from src.ranking import RankingConfig
from src.recommendation import (
    EMBEDDING_ONLY, FILTER_ONLY, FULL, PipelineConfig, Recommender, result_metrics, result_rows,
)
from src.retrieval import check_compatibility, load_index

OUT_DIR = PROJECT_ROOT / "data" / "processed" / "recommendation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)

env = describe_environment()
env

{'platform': 'macOS-26.6.2-arm64-arm-64bit',
 'processor': 'arm',
 'cpu_count': 12,
 'total_memory_gb': 32.0,
 'torch_version': '2.14.0',
 'cuda_available': False,
 'mps_available': True,
 'selected_device': 'mps'}

## 2. 저장된 임베딩 확인

manifest의 모델, 리비전, 원본 파일 해시를 현재 파일과 대조해 맞는 결과만 로드한다. 임베딩을 새로 만들지 않는다.

In [2]:
store = EmbeddingStore()
pd.DataFrame(store.list_results())

,name,model_id,model_revision,text_variant,num_vectors,dimension,created_at
0,embeddings_v1_textA_n5219.npy,NaN,NaN,NaN,5219,384,NaN
1,embeddings_v1_textB_n5219.npy,NaN,NaN,NaN,5219,384,NaN
2,multilingual-e5-base_d1287505_textA_v1_eb332ded,intfloat/multilingual-e5-base,d1287505,A,5219,768,2026-09-22T08:02:55+00:00
3,multilingual-e5-base_d1287505_textB_v1_f87e895f,intfloat/multilingual-e5-base,d1287505,B,5219,768,2026-09-22T08:03:45+00:00


In [3]:
sources = source_hashes()
index_a, ref_a = load_index("A", store, sources=sources)
index_b, ref_b = load_index("B", store, sources=sources)

for ref in (ref_a, ref_b):
    problems = check_compatibility(store.read_manifest(ref["name"]), sources=sources)
    print(f"텍스트 {ref['text_variant']}: {ref['name']}")
    print(f"  {ref['num_vectors']} x {ref['dimension']}, config_hash={ref['config_hash'][:12]}, "
          f"생성={ref['created_at']}, 호환 문제={problems or '없음'}, 후보범위={ref['후보범위']}")
assert index_a.size == index_b.size and index_a.size < 5219

텍스트 A: multilingual-e5-base_d1287505_textA_v1_eb332ded
  5219 x 768, config_hash=eb332dedac1f, 생성=2026-09-22T08:02:55+00:00, 호환 문제=없음, 후보범위={'프랜차이즈포함': False, '맨밥포함': False, '후보수': 1088, '전체수': 5219, '계열라벨': '채팅 0건, 나머지 키워드 규칙'}
텍스트 B: multilingual-e5-base_d1287505_textB_v1_f87e895f
  5219 x 768, config_hash=f87e895f6c65, 생성=2026-09-22T08:03:45+00:00, 호환 문제=없음, 후보범위={'프랜차이즈포함': False, '맨밥포함': False, '후보수': 1088, '전체수': 5219, '계열라벨': '채팅 0건, 나머지 키워드 규칙'}


## 3. 사용자 조건 파서 지원 범위

규칙 기반이며 아래 표가 지원 범위 전체다.

- 필수: 부정 표현("맵지 않은", "국물 없는")과 "꼭/무조건"이 붙은 표현. 허용값 집합이며 미확인 라벨은 충족하지 못한다.
- 선호: 긍정 표현. 점수에만 반영한다. 뒤에 거부어가 오면("따뜻한 거 싫어") 필수 제외가 된다.
- 기록만: 이중 부정, 허용 표현, 스키마에 없는 맛, 날씨와 기분.
- 메뉴 언급("치킨", "면")은 가점과 상한 면제에 쓰고, "피자 말고"는 그 메뉴를 걸러낸다.

In [4]:
pd.DataFrame(support_table())

,종류,규칙,예시,조건,비고
0,unhandled,이중부정,안 매운 건 싫어,-,이중 부정은 방향을 확정하지 않음
1,unhandled,허용표현,"매운 것도 괜찮아, 매워도 돼",-,허용·관용 표현은 필수·선호 조건으로 확정하지 않음
2,unhandled,시원한국물,시원한 국물,-,'시원한 국물'은 온도가 아닐 수 있어 확정하지 않음
3,unhandled,스키마외맛,단짠단짠한 음식,-,"라벨 스키마에 없는 맛·식감, 임베딩 유사도에만 맡김"
4,hard,매운맛_강함제외,"너무 맵지 않은, 너무 매운 거 싫어",매운맛=없음/약함/보통,-
5,hard,매운맛_제외,"맵지 않은, 안 매운, 매운 거 싫어",매운맛=없음,-
6,hard,국물_제외,"국물 없는, 국물이 없는, 국물 빼고",국물=국물없음,-
7,hard,뜨거움_제외,뜨겁지 않은,제공온도=따뜻함/상온/차가움,-
8,hard,차가움_제외,차갑지 않은,제공온도=뜨거움/따뜻함/상온,-
9,hard,기름짐_제외,"느끼하지 않은, 기름기 적은",기름짐=낮음/보통,-


In [5]:
BASE_QUERIES = [
    "비 오는 날 얼큰한 국물 먹고 싶어",
    "맵지 않고 따뜻한 음식",
    "차갑고 가볍게 먹을 메뉴",
    "바삭하고 기름진 음식",
    "든든한 밥 한 끼",
    "국물 없는 매운 음식",
    "상큼하고 시원한 음식",
    "포만감 있는 저녁밥",
    "빠르게 먹을 수 있는 간식",
    "따뜻한 국이나 찌개",
    "느끼하지 않은 담백한 음식",
    "단짠단짠한 음식",
]
EXTRA_QUERIES = {
    "부정": ["매운 거 싫어", "튀김 말고 구운 치킨"],
    "복합": ["피자 먹고 싶은데 느끼하지 않은 걸로", "차가운 국물 요리", "너무 맵지 않은 국물 요리"],
    "모순": ["맵지 않은 매운 음식", "국물 없는 국물 요리"],
    "미확정": ["안 매운 건 싫어", "매운 것도 괜찮아"],
    "빈 입력": [""],
}
ALL_QUERIES = BASE_QUERIES + [q for qs in EXTRA_QUERIES.values() for q in qs]
QUERY_KIND = {q: "기존" for q in BASE_QUERIES}
QUERY_KIND.update({q: kind for kind, qs in EXTRA_QUERIES.items() for q in qs})


def parse_row(q):
    p = parse_query(q)
    fmt = lambda conds: "; ".join(f"{c.attribute}={'/'.join(c.allowed)} ({c.evidence})" for c in conds) or "-"
    return {
        "유형": QUERY_KIND[q], "질의": q or "(빈 입력)", "필수": fmt(p.hard), "선호": fmt(p.soft),
        "메뉴언급": ", ".join(p.menu_terms) or "-",
        "미처리": "; ".join(f"{u['expression']} ({u['reason']})" for u in p.unhandled) or "-",
        "무시": ", ".join(i["expression"] for i in p.ignored) or "-",
        "모순": "; ".join(f"{c['attribute']}: {c['evidence']}" for c in p.contradictions) or "-",
    }


parsed_df = pd.DataFrame([parse_row(q) for q in ALL_QUERIES])
parsed_df

,유형,질의,필수,선호,메뉴언급,미처리,무시,모순
0,기존,비 오는 날 얼큰한 국물 먹고 싶어,-,매운맛=보통/강함 (얼큰한); 제공온도=뜨거움/따뜻함 (얼큰한); 국물=국물요리 (국물),-,-,비 오,-
1,기존,맵지 않고 따뜻한 음식,매운맛=없음 (맵지 않고),제공온도=뜨거움/따뜻함 (따뜻한),-,-,-,-
2,기존,차갑고 가볍게 먹을 메뉴,-,제공온도=차가움 (차갑고); 든든함=가벼움 (가볍게),-,-,-,-
3,기존,바삭하고 기름진 음식,-,기름짐=높음 (기름진); 조리법=튀김 (바삭하고),-,-,-,-
4,기존,든든한 밥 한 끼,-,든든함=든든함 (든든한); 계열=한식 (밥),밥,-,-,-
5,기존,국물 없는 매운 음식,국물=국물없음 (국물 없는),매운맛=보통/강함 (매운),-,-,-,-
6,기존,상큼하고 시원한 음식,-,제공온도=차가움 (시원한),-,"상큼 (라벨 스키마에 없는 맛·식감, 임베딩 유사도에만 맡김)",-,-
7,기존,포만감 있는 저녁밥,-,든든함=든든함 (포만감),-,-,저녁,-
8,기존,빠르게 먹을 수 있는 간식,-,든든함=가벼움 (간식),-,-,-,-
9,기존,따뜻한 국이나 찌개,-,"국물=국물요리 (국이, 찌개); 제공온도=뜨거움/따뜻함 (따뜻한)",찌개,-,-,-


## 4. 질의 임베딩 모델 로드

문서와 같은 모델로 사용자 문장만 `query: ` 접두어를 붙여 임베딩한다.

In [6]:
t0 = time.time()
embedder = E5Embedder(spec=DEFAULT_SPEC)
print(f"모델 로드 {time.time() - t0:.1f}초, device={embedder.device}, batch={embedder.batch_size}, "
      f"revision={DEFAULT_SPEC.revision[:8]}")
assert ref_b["model_revision"] == DEFAULT_SPEC.revision

encode_query = lambda text: embedder.encode_queries([text], show_progress=False)[0]
rec_b = Recommender(index_b, encode_query, ref_b)
rec_a = Recommender(index_a, encode_query, ref_a)
_ = rec_b.recommend("워밍업")  # 첫 MPS 호출 지연을 측정에서 제외

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

모델 로드 7.4초, device=mps, batch=32, revision=d1287505


/Users/jack/project/Menu-recommend-algorithmn/src/embedding/embedder.py:123: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  loaded_dim = self.model.get_sentence_embedding_dimension()


## 5. 비교 모드 설정

최종점수 = similarity_weight x 유사도 + preference_weight x 선호점수 + menu_match_weight x 메뉴일치.

- 임베딩만: 필터와 재랭킹 없음 (4단계와 같음)
- 조건적용: 필수 조건 필터만
- 조건+재랭킹+중복제어: 필터, 선호 재랭킹, 중복 제거, 메뉴군 상한 2, 선호 일치가 부족하면 후보를 400까지 확장

In [7]:
MODES = {"임베딩만": EMBEDDING_ONLY, "조건적용": FILTER_ONLY, "조건+재랭킹+중복제어": FULL}
pd.DataFrame({
    name: {**{k: v for k, v in asdict(cfg).items() if k != "ranking"}, **asdict(cfg.ranking)}
    for name, cfg in MODES.items()
})

,임베딩만,조건적용,조건+재랭킹+중복제어
top_k,5,5,5
candidate_k,100,100,100
preference_widen_k,400,400,400
apply_filters,False,True,True
similarity_weight,1.0,1.0,0.7
preference_weight,0.0,0.0,0.3
menu_match_weight,0.0,0.0,0.15
group_key,대표식품명,대표식품명,대표식품명
group_cap,0,0,2
group_penalty,0.0,0.0,0.0


## 6. 단일 질의 상세 흐름

조건 추출, 후보 100개 검색, 필터, 재랭킹, 중복 제어 순서로 Top-5를 만든다.

In [8]:
SHOW_COLS = ["순위", "메뉴명", "업체명", "대표식품명", "주요라벨", "유사도", "선호점수", "최종점수", "추천근거"]


def show(result):
    print(f"질의: {result['질의']!r}  상태={result['상태']}  사유={result['사유']}")
    print(f"검색범위={result['검색범위']} (확장사유={result['확장사유']})  후보={result['후보수']}  "
          f"필터통과={result['필터통과']}  필터제외={result['필터제외']} {result['필터제외사유']}")
    print(f"반환 {result['반환수']}/{result['요청수']}  실행시간 {result['실행시간'].get('전체', 0):.3f}초  "
          f"임베딩={result['임베딩']['name']}")
    return pd.DataFrame(result["추천"], columns=SHOW_COLS)


detail = rec_b.recommend("국물 없는 매운 음식")
print("조건:", parse_query(detail["질의"]).summary())
show(detail)

조건: 필수 국물=국물없음 (국물 없는) | 선호 매운맛=보통/강함 (매운)
질의: '국물 없는 매운 음식'  상태=ok  사유=None
검색범위=[100, 200, 400] (확장사유=선호 조건 일치 후보 부족)  후보=400  필터통과=84  필터제외=316 {'국물=국물요리': 258, '국물=국물약간': 58}
반환 5/5  실행시간 0.017초  임베딩=multilingual-e5-base_d1287505_textB_v1_f87e895f


,순위,메뉴명,업체명,대표식품명,주요라벨,유사도,선호점수,최종점수,추천근거
0,1,쟁반국수,-,쟁반국수,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식",0.8543,1.0,0.8980,유사도 0.8543 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)
1,2,막국수,-,막국수,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식",0.8497,1.0,0.8948,유사도 0.8497 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)
2,3,국수 쟁반막국수,-,국수,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식",0.8475,1.0,0.8933,유사도 0.8475 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)
3,4,황태구이,-,황태구이,"매운맛 보통, 국물없음, 제공온도 뜨거움, 조리법 구이, 기름짐 낮음, 한식, 안주",0.8426,1.0,0.8898,유사도 0.8426 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)
4,5,해물볶음,-,해물볶음,"매운맛 보통, 국물없음, 제공온도 따뜻함, 조리법 볶음, 기름짐 보통, 든든함 보통, 한식",0.8422,1.0,0.8896,유사도 0.8422 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)


In [9]:
# 중복·상한으로 빠진 후보
pd.DataFrame(detail["제외"])

""


### 후보 부족 시 동작

필수 조건 통과 항목이 부족하면 전체까지, 선호 일치 항목이 부족하면 400까지 검색 범위를 넓힌다. 필수 조건은 완화하지 않는다.
첫 예는 확장으로 채워지는 경우, 둘째 예는 60개를 요청해 전체를 훑어도 부족한 경우다.

In [10]:
widened = rec_b.recommend("꼭 차가운 찜 요리")
print("조건:", parse_query(widened["질의"]).summary())
show(widened)

조건: 필수 제공온도=차가움 (차가운) | 선호 조리법=찜 (찜)
질의: '꼭 차가운 찜 요리'  상태=ok  사유=None
검색범위=[100, 200, 400] (확장사유=선호 조건 일치 후보 부족)  후보=400  필터통과=25  필터제외=375 {'제공온도=뜨거움': 335, '제공온도=따뜻함': 36, '제공온도=미확인': 4}
반환 5/5  실행시간 0.015초  임베딩=multilingual-e5-base_d1287505_textB_v1_f87e895f


,순위,메뉴명,업체명,대표식품명,주요라벨,유사도,선호점수,최종점수,추천근거
0,1,냉면 회냉면 홍어,-,냉면,"매운맛 보통, 국물약간, 제공온도 차가움, 조리법 혼합, 기름짐 낮음, 든든함 보통, 한식",0.8359,0.0,0.5851,유사도 0.8359 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=혼합 불일치(찜)
1,2,가지냉국,-,가지냉국,"매운맛 없음, 국물요리, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 가벼움, 한식",0.8352,0.0,0.5846,유사도 0.8352 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=비조리 불일치(찜)
2,3,콩국수,-,콩국수,"매운맛 없음, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 보통, 든든함 보통, 한식",0.8351,0.0,0.5845,유사도 0.8351 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=끓임 불일치(찜)
3,4,연어롤,-,연어롤,"매운맛 없음, 국물없음, 제공온도 차가움, 조리법 비조리, 기름짐 보통, 일식",0.8351,0.0,0.5845,유사도 0.8351 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=비조리 불일치(찜)
4,5,물냉면,-,물냉면,"매운맛 없음, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식",0.8348,0.0,0.5844,유사도 0.8348 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=끓임 불일치(찜)


In [11]:
short = rec_b.recommend("무조건 차가운 튀김", PipelineConfig(top_k=60))
print("조건:", parse_query(short["질의"]).summary())
show(short).tail(5)

조건: 필수 제공온도=차가움 (차가운) | 선호 조리법=튀김 (튀김)
질의: '무조건 차가운 튀김'  상태=shortage  사유=전체 1088건 중 필수 조건 통과 47건, 중복·상한 제외 후 31건만 남음
검색범위=[100, 200, 400, 800, 1088] (확장사유=필수 조건 통과 후보 부족)  후보=1088  필터통과=47  필터제외=1041 {'제공온도=뜨거움': 804, '제공온도=따뜻함': 183, '제공온도=상온': 49, '제공온도=미확인': 5}
반환 31/60  실행시간 0.025초  임베딩=multilingual-e5-base_d1287505_textB_v1_f87e895f


,순위,메뉴명,업체명,대표식품명,주요라벨,유사도,선호점수,최종점수,추천근거
26,27,회덮밥,-,회덮밥,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 보통, 일식",0.8082,0.0,0.5657,유사도 0.8082 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=비조리 불일치(튀김)
27,28,우럭회덮밥 양념장,-,우럭회덮밥,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 보통, 일식",0.8015,0.0,0.5611,유사도 0.8015 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=비조리 불일치(튀김)
28,29,밀면 물밀면,-,밀면,"매운맛 약함, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식",0.8008,0.0,0.5606,유사도 0.8008 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=끓임 불일치(튀김)
29,30,묵말이 도토리묵,-,묵말이,"매운맛 없음, 국물요리, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 가벼움, 한식",0.7980,0.0,0.5586,유사도 0.7980 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=비조리 불일치(튀김)
30,31,묵국 메밀묵,-,묵국,"매운맛 약함, 국물요리, 제공온도 차가움, 조리법 혼합, 기름짐 낮음, 든든함 가벼움, 일식",0.7954,0.0,0.5568,유사도 0.7954 / 필수 제공온도=차가움 충족(차가운) / 선호 조리법=혼합 불일치(튀김)


## 7. 비교 실험 (텍스트 B)

4단계 질의 12개에 부정, 복합, 모순, 미확정, 빈 입력 사례를 더해 세 모드를 비교한다. 지표는 저장된 추정 라벨 기준이다.

- 조건위반수: 필수 조건에 맞지 않는 반환 항목 수 (미확인 포함)
- 미확인포함수: 질의가 언급한 속성이 미확인인 항목 수
- 선호불일치수: 선호에 맞지 않는 항목과 조건 쌍의 수
- 중복메뉴수, 대표식품명반복수, 메뉴군반복수: 같은 메뉴 변형과 같은 군의 반복 수

In [12]:
def run_all(rec, queries, modes, tag):
    rows, metrics, results = [], [], {}
    for q in queries:
        for mode, cfg in modes.items():
            r = rec.recommend(q, cfg)
            results[(mode, q)] = r
            rows += result_rows(r, 텍스트구성=tag, 모드=mode, 유형=QUERY_KIND[q])
            metrics.append({"텍스트구성": tag, "모드": mode, "유형": QUERY_KIND[q], **result_metrics(r)})
    return pd.DataFrame(rows), pd.DataFrame(metrics), results


t0 = time.time()
rows_b, metrics_b, results_b = run_all(rec_b, ALL_QUERIES, MODES, "B")
print(f"{len(ALL_QUERIES)}개 질의 x {len(MODES)}개 모드 = {len(metrics_b)}회 실행, 추천 행 {len(rows_b)}개, {time.time() - t0:.1f}초")

22개 질의 x 3개 모드 = 66회 실행, 추천 행 285개, 0.3초


In [13]:
# 질의별 메뉴명·업체·주요 라벨·점수·근거 (전체 파이프라인)
rows_b[rows_b["모드"] == "조건+재랭킹+중복제어"][["유형", "질의", *SHOW_COLS]]

,유형,질의,순위,메뉴명,업체명,대표식품명,주요라벨,유사도,선호점수,최종점수,추천근거
10,기존,비 오는 날 얼큰한 국물 먹고 싶어,1,수제비 김치,-,수제비,"매운맛 보통, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식",0.8089,1.0000,1.0162,유사도 0.8089 / 선호 매운맛=보통 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물) / 언급 메뉴 수제비 일...
11,기존,비 오는 날 얼큰한 국물 먹고 싶어,2,칼국수,-,칼국수,"매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함, 한식",0.8178,0.6667,0.9225,유사도 0.8178 / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물) / 선호 매운맛=없음 불일치(얼큰한) / 언급 메뉴 칼국수 ...
12,기존,비 오는 날 얼큰한 국물 먹고 싶어,3,김치전,-,김치전,"매운맛 보통, 국물없음, 제공온도 따뜻함, 조리법 부침, 기름짐 보통, 든든함 보통, 한식, 안주",0.8069,0.6667,0.9148,유사도 0.8069 / 선호 매운맛=보통 일치(얼큰한) / 선호 제공온도=따뜻함 일치(얼큰한) / 선호 국물=국물없음 불일치(국물) / 언급 메뉴 전·적 ...
13,기존,비 오는 날 얼큰한 국물 먹고 싶어,4,해장국 뼈다귀,-,해장국,"매운맛 보통, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 보통, 든든함, 한식",0.8271,1.0000,0.8789,유사도 0.8271 / 선호 매운맛=보통 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)
14,기존,비 오는 날 얼큰한 국물 먹고 싶어,5,꽃게 매운탕,-,꽃게 매운탕,"매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식",0.8248,1.0000,0.8774,유사도 0.8248 / 선호 매운맛=강함 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)
25,기존,맵지 않고 따뜻한 음식,1,화양적,-,화양적,"매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 부침, 기름짐 보통, 한식, 안주",0.8407,1.0000,0.8885,유사도 0.8407 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=따뜻함 일치(따뜻한)
26,기존,맵지 않고 따뜻한 음식,2,무 된장국,-,무 된장국,"매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움, 한식",0.8398,1.0000,0.8878,유사도 0.8398 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=뜨거움 일치(따뜻한)
27,기존,맵지 않고 따뜻한 음식,3,햄버거,-,햄버거,"매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 혼합, 기름짐 높음, 든든함, 양식",0.8397,1.0000,0.8878,유사도 0.8397 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=따뜻함 일치(따뜻한)
28,기존,맵지 않고 따뜻한 음식,4,복지리,-,복지리,"매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 한식",0.8396,1.0000,0.8877,유사도 0.8396 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=뜨거움 일치(따뜻한)
29,기존,맵지 않고 따뜻한 음식,5,족발,-,족발,"매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 조림, 기름짐 높음, 든든함, 한식, 안주",0.8390,1.0000,0.8873,유사도 0.8390 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=따뜻함 일치(따뜻한)


In [14]:
# 세 모드 나란히 보기: 메뉴명 [업체] 최종점수
def compact(rows, modes):
    rows = rows.copy()
    rows["항목"] = rows["메뉴명"] + " [" + rows["업체명"] + "] " + rows["최종점수"].map("{:.3f}".format)
    return rows.pivot(index=["질의", "순위"], columns="모드", values="항목")[list(modes)].fillna("")


compact(rows_b, MODES)

모드                                         임베딩만                    조건적용                 조건+재랭킹+중복제어
질의                   순위                                                                            
국물 없는 매운 음식          1         해물 매운탕 [-] 0.866          쟁반국수 [-] 0.854              쟁반국수 [-] 0.898
                     2         광어 매운탕 [-] 0.865         모듬 김밥 [-] 0.850               막국수 [-] 0.895
                     3         잉어 매운탕 [-] 0.865           막국수 [-] 0.850          국수 쟁반막국수 [-] 0.893
                     4         메기 매운탕 [-] 0.864         해물 덮밥 [-] 0.849              황태구이 [-] 0.890
                     5         버섯 매운탕 [-] 0.863            족발 [-] 0.848              해물볶음 [-] 0.890
너무 맵지 않은 국물 요리       1         해물 된장국 [-] 0.846        해물 된장국 [-] 0.846            해물 된장국 [-] 0.892
                     2          굴 미역국 [-] 0.846         굴 미역국 [-] 0.846             굴 미역국 [-] 0.892
                     3            무국물 [-] 0.846           무국물 [-] 0.846               무국물 [-] 0.892
                     4          무 된장국 [-] 0.845         무 된장국 [-] 0.845             무 된장국 [-] 0.891
                     5            선짓국 [-] 0.845           선짓국 [-] 0.845               선짓국 [-] 0.891
느끼하지 않은 담백한 음식       1            화양적 [-] 0.842           화양적 [-] 0.842             무 된장국 [-] 0.887
                     2         붕어 매운탕 [-] 0.842        붕어 매운탕 [-] 0.842             홍합 무국 [-] 0.887
                     3           오리백숙 [-] 0.841          오리백숙 [-] 0.841               백합죽 [-] 0.887
                     4         명태 매운탕 [-] 0.841        명태 매운탕 [-] 0.841               무국물 [-] 0.886
                     5         꽃게 매운탕 [-] 0.841        꽃게 매운탕 [-] 0.841               채소죽 [-] 0.886
단짠단짠한 음식             1             분짜 [-] 0.838            분짜 [-] 0.838                분짜 [-] 0.586
                     2             짬뽕 [-] 0.838            짬뽕 [-] 0.838                짬뽕 [-] 0.586
                     3         붕어 매운탕 [-] 0.837        붕어 매운탕 [-] 0.837            붕어 매운탕 [-] 0.586
                     4             닭찜 [-] 0.837            닭찜 [-] 0.837                닭찜 [-] 0.586
                     5         소고기 떡찜 [-] 0.836        소고기 떡찜 [-] 0.836            소고기 떡찜 [-] 0.585
든든한 밥 한 끼            1            잡탕밥 [-] 0.832           잡탕밥 [-] 0.832             육회비빔밥 [-] 1.026
                     2             김밥 [-] 0.827            김밥 [-] 0.827            덮밥 닭고기 [-] 1.026
                     3           삼각김밥 [-] 0.826          삼각김밥 [-] 0.826            소고기 덮밥 [-] 1.025
                     4         닭고기 덮밥 [-] 0.825        닭고기 덮밥 [-] 0.825            비빔 잡곡밥 [-] 1.024
                     5            잡채밥 [-] 0.824           잡채밥 [-] 0.824           돼지고기 덮밥 [-] 1.022
따뜻한 국이나 찌개           1           두부찌개 [-] 0.865          두부찌개 [-] 0.865              두부찌개 [-] 1.055
                     2       감자 소고기찌개 [-] 0.863      감자 소고기찌개 [-] 0.863          감자 소고기찌개 [-] 1.054
                     3     섞어찌개(모듬찌개) [-] 0.863    섞어찌개(모듬찌개) [-] 0.863        섞어찌개(모듬찌개) [-] 1.054
                     4         굴 두부찌개 [-] 0.863        굴 두부찌개 [-] 0.863            굴 두부찌개 [-] 1.054
                     5       두부찌개 바지락 [-] 0.862      두부찌개 바지락 [-] 0.862              조기찌개 [-] 1.052
매운 거 싫어              1          복 매운탕 [-] 0.814            미음 [-] 0.802                미음 [-] 0.561
                     2         버섯 매운탕 [-] 0.814         모듬 초밥 [-] 0.796             모듬 초밥 [-] 0.557
                     3        매운탕 짱뚱어 [-] 0.814            묵밥 [-] 0.796                묵밥 [-] 0.557
                     4         잉어 매운탕 [-] 0.813        대합 미역국 [-] 0.796            대합 미역국 [-] 0.557
                     5         해물 매운탕 [-] 0.813        함박스테이크 [-] 0.795            함박스테이크 [-] 0.556
매운 것도 괜찮아            1         해물 매운탕 [-] 0.834        해물 매운탕 [-] 0.834            해물 매운탕 [-] 0.583
                     2         메기 매운탕 [-] 0.833        메기 매운탕 [-] 0.833            메기 매운탕 [-] 0.583
                     3          복 매운탕 [-] 0.833         복 매운탕 [-] 0.833  닭볶음(닭갈비) 매운양념 치즈 [-] 0.578


In [15]:
METRIC_COLS = ["반환수", "조건위반수", "미확인포함수", "선호불일치수", "중복메뉴수", "대표식품명반복수", "메뉴군반복수", "검색범위", "실행시간초"]
metrics_b.pivot(index=["유형", "질의"], columns="모드", values=METRIC_COLS[:7]).reindex(columns=list(MODES), level=1)

반환수                  조건위반수                  미확인포함수                  선호불일치수  ...             중복메뉴수                  대표식품명반복수                  메뉴군반복수                 
모드                        임베딩만 조건적용 조건+재랭킹+중복제어  임베딩만 조건적용 조건+재랭킹+중복제어   임베딩만 조건적용 조건+재랭킹+중복제어   임베딩만  ... 조건+재랭킹+중복제어  임베딩만 조건적용 조건+재랭킹+중복제어     임베딩만 조건적용 조건+재랭킹+중복제어   임베딩만 조건적용 조건+재랭킹+중복제어
유형   질의                                                                                                ...                                                                                     
기존   국물 없는 매운 음식             5    5           5     5    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      4    0           0
     느끼하지 않은 담백한 음식          5    5           5     0    0           0      0    0           0      5  ...           0     0    0           0        0    0           0      2    2           0
     단짠단짠한 음식                5    5           5     0    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      0    0           0
     든든한 밥 한 끼               5    5           5     0    0           0      0    0           0      8  ...           0     0    0           0        0    0           0      0    0           2
     따뜻한 국이나 찌개              5    5           5     0    0           0      0    0           0      0  ...           0     1    1           0        1    1           0      2    2           1
     맵지 않고 따뜻한 음식            5    5           5     2    0           0      2    0           0      1  ...           0     0    0           0        0    0           0      0    0           0
     바삭하고 기름진 음식             5    5           5     0    0           0      0    0           0     10  ...           0     0    0           0        0    0           0      3    3           0
     비 오는 날 얼큰한 국물 먹고 싶어     5    5           5     0    0           0      0    0           0      5  ...           2     0    0           0        0    0           0      0    0           0
     빠르게 먹을 수 있는 간식          5    5           5     0    0           0      1    1           0      4  ...           0     1    1           0        1    1           0      1    1           0
     상큼하고 시원한 음식             5    5           5     0    0           0      0    0           0      5  ...           0     0    0           0        0    0           1      1    1           1
     차갑고 가볍게 먹을 메뉴           5    5           5     0    0           0      2    2           0      8  ...           3     0    0           0        0    0           0      0    0           0
     포만감 있는 저녁밥              5    5           5     0    0           0      0    0           0      4  ...           0     0    0           0        0    0           0      2    2           1
모순   국물 없는 국물 요리             0    0           0     0    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      0    0           0
     맵지 않은 매운 음식             0    0           0     0    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      0    0           0
미확정  매운 것도 괜찮아               5    5           5     0    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      4    4           1
     안 매운 건 싫어               5    5           5     0    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      4    4           1
복합   너무 맵지 않은 국물 요리          5    5           5     0    0           0      0    0           0      0  ...           0     0    0           0        0    0           0      1    1           1
     차가운 국물 요리               5    5           5     0    0           0      0    0           0      5  ...           0     0    0           0        0    0           0      0    0        

In [16]:
AGG = {"반환수": "sum", "조건위반수": "sum", "미확인포함수": "sum", "선호불일치수": "sum",
       "중복메뉴수": "sum", "대표식품명반복수": "sum", "메뉴군반복수": "sum", "실행시간초": "mean"}
summary_b = metrics_b.groupby("모드").agg(AGG).loc[list(MODES)]
summary_b

,반환수,조건위반수,미확인포함수,선호불일치수,중복메뉴수,대표식품명반복수,메뉴군반복수,실행시간초
모드,,,,,,,,
임베딩만,95,19,5,58,3,7,33,0.010391
조건적용,95,0,3,58,2,3,22,0.000423
조건+재랭킹+중복제어,95,0,0,5,0,1,9,0.000891


In [17]:
# 정상 반환이 아닌 질의: 모순, 빈 입력, 후보 부족
mask = (metrics_b["모드"] == "조건+재랭킹+중복제어") & (metrics_b["상태"] != "ok")
for _, m in metrics_b[mask].iterrows():
    r = results_b[("조건+재랭킹+중복제어", m["질의"])]
    print(f"[{m['상태']}] {m['질의']!r}: 반환 {m['반환수']}/{m['요청수']} — {r['사유']}")

[contradiction] '맵지 않은 매운 음식': 반환 0/5 — 필수 조건이 서로 모순됨: 매운맛(맵지 않은 vs 매운)
[contradiction] '국물 없는 국물 요리': 반환 0/5 — 필수 조건이 서로 모순됨: 국물(국물 없는 vs 국물)
[empty_query] '': 반환 0/5 — 입력이 비어 있음


## 8. 텍스트 A 기준선

A(음식명+분류)와 B(A+속성 라벨)를 같은 파이프라인으로 비교한다. 우열은 단정하지 않는다.

In [18]:
rows_a, metrics_a, results_a = run_all(rec_a, ALL_QUERIES, {"임베딩만": EMBEDDING_ONLY, "조건+재랭킹+중복제어": FULL}, "A")
summary_ab = pd.concat([metrics_a, metrics_b]).groupby(["텍스트구성", "모드"]).agg(AGG)
summary_ab

반환수  조건위반수  미확인포함수  선호불일치수  중복메뉴수  대표식품명반복수  메뉴군반복수     실행시간초
텍스트구성 모드                                                                        
A     임베딩만          95     20       3      54      8        10      26  0.010909
      조건+재랭킹+중복제어   95      0       0       3      0         1       9  0.000818
B     임베딩만          95     19       5      58      3         7      33  0.010391
      조건+재랭킹+중복제어   95      0       0       5      0         1       9  0.000891
      조건적용          95      0       3      58      2         3      22  0.000423

In [19]:
# A/B 전체 파이프라인 1순위가 같은 질의 수
valid = [q for q in ALL_QUERIES if results_b[("조건+재랭킹+중복제어", q)]["추천"]]
same = [q for q in valid
        if results_a[("조건+재랭킹+중복제어", q)]["추천"] and
        results_a[("조건+재랭킹+중복제어", q)]["추천"][0]["라벨링단위ID"] == results_b[("조건+재랭킹+중복제어", q)]["추천"][0]["라벨링단위ID"]]
print(f"A/B 1순위 동일: {len(same)}/{len(valid)}건")
for q in valid:
    a = results_a[("조건+재랭킹+중복제어", q)]["추천"]
    b = results_b[("조건+재랭킹+중복제어", q)]["추천"]
    print(f"  {q!r}: A={a[0]['메뉴명'] if a else '-'} / B={b[0]['메뉴명'] if b else '-'}")

A/B 1순위 동일: 5/19건
  '비 오는 날 얼큰한 국물 먹고 싶어': A=수제비 김치 / B=수제비 김치
  '맵지 않고 따뜻한 음식': A=볶음 우동 / B=화양적
  '차갑고 가볍게 먹을 메뉴': A=냉국 미역 오이 / B=미역냉국 오이 고추
  '바삭하고 기름진 음식': A=핫도그 / B=깐풍기
  '든든한 밥 한 끼': A=비빔 잡곡밥 / B=육회비빔밥
  '국물 없는 매운 음식': A=비빔국수 / B=쟁반국수
  '상큼하고 시원한 음식': A=쫄면 / B=국수 김치말이국수
  '포만감 있는 저녁밥': A=하이라이스 / B=소고기 덮밥
  '빠르게 먹을 수 있는 간식': A=땅콩죽 / B=주먹밥
  '따뜻한 국이나 찌개': A=국수전골 / B=두부찌개
  '느끼하지 않은 담백한 음식': A=무 된장국 / B=무 된장국
  '단짠단짠한 음식': A=짬뽕 / B=분짜
  '매운 거 싫어': A=미음 / B=미음
  '튀김 말고 구운 치킨': A=치킨데리야끼 / B=치킨데리야끼
  '피자 먹고 싶은데 느끼하지 않은 걸로': A=피자 불고기피자 / B=피자 불고기피자
  '차가운 국물 요리': A=미역냉국 / B=국수 김치말이국수
  '너무 맵지 않은 국물 요리': A=무 된장국 / B=해물 된장국
  '안 매운 건 싫어': A=미음 / B=복 매운탕
  '매운 것도 괜찮아': A=미음 / B=해물 매운탕


## 9. 중복·다양성 설정 실험

메뉴군 상한(0, 1, 2, 3)과 감점(0, 0.02)을 조합해 반복 수와 부족 질의 수를 본다.

In [20]:
def experiment(rec, queries, configs):
    out = []
    for name, cfg in configs.items():
        df = pd.DataFrame([result_metrics(rec.recommend(q, cfg)) for q in queries])
        out.append({
            "설정": name, "반환수합": df["반환수"].sum(), "조건위반합": df["조건위반수"].sum(),
            "미확인합": df["미확인포함수"].sum(), "선호불일치합": df["선호불일치수"].sum(),
            "중복메뉴합": df["중복메뉴수"].sum(), "대표식품명반복합": df["대표식품명반복수"].sum(),
            "메뉴군반복합": df["메뉴군반복수"].sum(), "확장질의수": int((df["확장횟수"] > 0).sum()),
            "부족질의수": int((df["상태"] == "shortage").sum()), "평균실행초": round(df["실행시간초"].mean(), 4),
        })
    return pd.DataFrame(out).set_index("설정")


VALID_QUERIES = [q for q in ALL_QUERIES if q and not parse_query(q).contradictions]

diversity_configs = {"중복제거·상한 없음": PipelineConfig(ranking=RankingConfig(collapse_duplicates=False, group_cap=0))}
for cap in (0, 1, 2, 3):
    for penalty in (0.0, 0.02):
        diversity_configs[f"상한{cap} 감점{penalty}"] = PipelineConfig(
            ranking=RankingConfig(group_cap=cap, group_penalty=penalty))
exp_diversity = experiment(rec_b, VALID_QUERIES, diversity_configs)
exp_diversity

,반환수합,조건위반합,미확인합,선호불일치합,중복메뉴합,대표식품명반복합,메뉴군반복합,확장질의수,부족질의수,평균실행초
설정,,,,,,,,,,
중복제거·상한 없음,95,0,0,7,6,10,24,5,0,0.0010
상한0 감점0.0,95,0,0,5,0,4,18,5,0,0.0009
상한0 감점0.02,95,0,0,5,0,0,3,5,0,0.0016
상한1 감점0.0,95,0,0,5,0,0,3,5,0,0.0010
상한1 감점0.02,95,0,0,5,0,0,3,5,0,0.0018
상한2 감점0.0,95,0,0,5,0,1,9,5,0,0.0010
상한2 감점0.02,95,0,0,5,0,0,3,5,0,0.0016
상한3 감점0.0,95,0,0,5,0,3,13,5,0,0.0010
상한3 감점0.02,95,0,0,5,0,0,3,5,0,0.0016


In [21]:
# 사용자가 '피자'를 직접 요청하면 상한을 면제한다
for name in ("상한1 감점0.0", "상한2 감점0.0"):
    r = rec_b.recommend("피자 먹고 싶은데 느끼하지 않은 걸로", diversity_configs[name])
    print(f"{name}: {[it['메뉴명'] for it in r['추천']]}")
r = rec_b.recommend("단짠단짠한 음식", FULL)
print("피자 언급 없는 질의:", [(it["메뉴명"], it["대표식품명"]) for it in r["추천"]])
print("  제외:", [(e["메뉴명"], e["제외사유"]) for e in r["제외"]])

상한1 감점0.0: ['피자 불고기피자', '돼지고기 피망잡채', '회덮밥', '기스면', '애호박죽']
상한2 감점0.0: ['피자 불고기피자', '돼지고기 피망잡채', '회덮밥', '기스면', '애호박죽']
피자 언급 없는 질의: [('분짜', '분짜'), ('짬뽕', '짬뽕'), ('붕어 매운탕', '붕어 매운탕'), ('닭찜', '닭찜'), ('소고기 떡찜', '소고기 떡찜')]
  제외: []


적용 기준: 중복 제거는 항상 켠다. 상한과 감점은 반복을 줄이지만 정답 기준 근거가 없어 설명이 가장 단순한 상한 2, 감점 0을 기본으로 둔다. 언급한 메뉴는 상한을 면제한다.

## 10. 선호 가중치 민감도

후보 안의 유사도 차이가 작아 가중치는 사실상 켜고 끄는 스위치로 동작한다.

In [22]:
def mean_similarity(rec, queries, cfg):
    sims = [it["유사도"] for q in queries for it in rec.recommend(q, cfg)["추천"]]
    return round(float(np.mean(sims)), 4)


weight_configs = {
    f"선호가중치 {w}": PipelineConfig(ranking=RankingConfig(similarity_weight=1 - w, preference_weight=w))
    for w in (0.0, 0.05, 0.1, 0.3, 0.5)
}
exp_weights = experiment(rec_b, VALID_QUERIES, weight_configs)
exp_weights["평균유사도"] = [mean_similarity(rec_b, VALID_QUERIES, c) for c in weight_configs.values()]
exp_weights

,반환수합,조건위반합,미확인합,선호불일치합,중복메뉴합,대표식품명반복합,메뉴군반복합,확장질의수,부족질의수,평균실행초,평균유사도
설정,,,,,,,,,,,
선호가중치 0.0,95,0,3,60,0,0,9,0,0,0.0006,0.8359
선호가중치 0.05,95,0,0,10,0,1,9,6,0,0.0010,0.8312
선호가중치 0.1,95,0,0,10,0,1,9,6,0,0.0010,0.8312
선호가중치 0.3,95,0,0,5,0,1,9,5,0,0.0009,0.8315
선호가중치 0.5,95,0,0,3,0,1,10,4,0,0.0008,0.8320


## 11. 후보 수 민감도

자동 확장을 끄고 후보 수를 고정한 뒤 기본 설정과 비교한다.

In [23]:
k_configs = {f"후보 {k} 고정": PipelineConfig(candidate_k=k, preference_widen_k=k) for k in (50, 100, 200, 400)}
k_configs["기본 (100->400 자동 확장)"] = FULL
exp_k = experiment(rec_b, VALID_QUERIES, k_configs)
exp_k["평균유사도"] = [mean_similarity(rec_b, VALID_QUERIES, c) for c in k_configs.values()]
exp_k

,반환수합,조건위반합,미확인합,선호불일치합,중복메뉴합,대표식품명반복합,메뉴군반복합,확장질의수,부족질의수,평균실행초,평균유사도
설정,,,,,,,,,,,
후보 50 고정,95,0,0,22,0,0,11,1,0,0.0006,0.8342
후보 100 고정,95,0,0,21,0,0,10,0,0,0.0006,0.8336
후보 200 고정,95,0,0,15,0,0,9,0,0,0.0008,0.8331
후보 400 고정,95,0,0,5,0,1,9,0,0,0.0012,0.8315
기본 (100->400 자동 확장),95,0,0,5,0,1,9,5,0,0.0010,0.8315


In [24]:
# 후보 100개 안에 선호 일치 항목이 없던 질의: 고정 100 vs 기본(자동 확장)
for name in ("후보 100 고정", "기본 (100->400 자동 확장)"):
    r = rec_b.recommend("차갑고 가볍게 먹을 메뉴", k_configs[name])
    print(f"{name}: 검색범위={r['검색범위']} 확장사유={r['확장사유']}")
    print("   ", [(it["메뉴명"], it["선호점수"]) for it in r["추천"]])

후보 100 고정: 검색범위=[100] 확장사유=None
    [('소고기 감자죽', 0.5), ('미소된장국', 0.5), ('국수 김치말이국수', 0.5), ('주먹밥', 0.5), ('채소죽 달걀', 0.5)]
기본 (100->400 자동 확장): 검색범위=[100, 200, 400] 확장사유=선호 조건 일치 후보 부족
    [('미역냉국 오이 고추', 1.0), ('냉국 미역 오이', 1.0), ('소고기 감자죽', 0.5), ('미소된장국', 0.5), ('국수 김치말이국수', 0.5)]


## 12. 개선 사례: 임베딩만 vs 전체 파이프라인

In [25]:
def side_by_side(q):
    fmt = lambda r: [f"{it['메뉴명']} [{it['업체명']}] — {it['주요라벨']}" for it in r["추천"]]
    left, right = fmt(results_b[("임베딩만", q)]), fmt(results_b[("조건+재랭킹+중복제어", q)])
    n = max(len(left), len(right))
    return pd.DataFrame({"임베딩만": left + [""] * (n - len(left)),
                         "조건+재랭킹+중복제어": right + [""] * (n - len(right))}, index=range(1, n + 1))


for q in ["맵지 않고 따뜻한 음식", "국물 없는 매운 음식", "차가운 국물 요리", "단짠단짠한 음식", "느끼하지 않은 담백한 음식"]:
    print(f"\n### {q}  |  {parse_query(q).summary()}")
    display(side_by_side(q))


### 맵지 않고 따뜻한 음식  |  필수 매운맛=없음 (맵지 않고) | 선호 제공온도=뜨거움/따뜻함 (따뜻한)


,임베딩만,조건+재랭킹+중복제어
1,"소고기채소볶음 [-] — 국물없음, 제공온도 따뜻함, 조리법 볶음, 기름짐 보통, 든든함 보통, 한식","화양적 [-] — 매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 부침, 기름짐 보통, 한식, 안주"
2,"돼지고기 덮밥 [-] — 국물없음, 제공온도 따뜻함, 조리법 볶음, 기름짐 보통, 든든함, 한식","무 된장국 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움, 한식"
3,"화양적 [-] — 매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 부침, 기름짐 보통, 한식, 안주","햄버거 [-] — 매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 혼합, 기름짐 높음, 든든함, 양식"
4,"샌드위치 햄 치즈 채소 [-] — 매운맛 없음, 국물없음, 제공온도 상온, 조리법 비조리, 기름짐 보통, 양식","복지리 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 한식"
5,"무 된장국 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움, 한식","족발 [-] — 매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 조림, 기름짐 높음, 든든함, 한식, 안주"



### 국물 없는 매운 음식  |  필수 국물=국물없음 (국물 없는) | 선호 매운맛=보통/강함 (매운)


,임베딩만,조건+재랭킹+중복제어
1,"해물 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","쟁반국수 [-] — 매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식"
2,"광어 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","막국수 [-] — 매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식"
3,"잉어 매운탕 [-] — 매운맛 보통, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 한식","국수 쟁반막국수 [-] — 매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식"
4,"메기 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","황태구이 [-] — 매운맛 보통, 국물없음, 제공온도 뜨거움, 조리법 구이, 기름짐 낮음, 한식, 안주"
5,"버섯 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 한식","해물볶음 [-] — 매운맛 보통, 국물없음, 제공온도 따뜻함, 조리법 볶음, 기름짐 보통, 든든함 보통, 한식"



### 차가운 국물 요리  |  선호 국물=국물요리 (국물) | 선호 제공온도=차가움 (차가운)


,임베딩만,조건+재랭킹+중복제어
1,"게국지 [-] — 매운맛 보통, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","국수 김치말이국수 [-] — 매운맛 보통, 국물요리, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 보통, 한식"
2,"국수 막국수 [-] — 매운맛 약함, 국물약간, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","콩국수 [-] — 매운맛 없음, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 보통, 든든함 보통, 한식"
3,"냉이 된장국 [-] — 매운맛 약함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움, 한식","냉국 미역 [-] — 매운맛 없음, 국물요리, 제공온도 차가움, 조리법 혼합, 기름짐 낮음, 한식"
4,"선짓국 [-] — 매운맛 약함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 한식","냉면 열무냉면 [-] — 매운맛 약함, 국물요리, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식"
5,"해물 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","가지냉국 [-] — 매운맛 없음, 국물요리, 제공온도 차가움, 조리법 비조리, 기름짐 낮음, 든든함 가벼움, 한식"



### 단짠단짠한 음식  |  미처리 '단짠'


,임베딩만,조건+재랭킹+중복제어
1,"분짜 [-] — 매운맛 약함, 국물약간, 제공온도 따뜻함, 조리법 혼합, 기름짐 보통, 든든함 보통, 동남아","분짜 [-] — 매운맛 약함, 국물약간, 제공온도 따뜻함, 조리법 혼합, 기름짐 보통, 든든함 보통, 동남아"
2,"짬뽕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 보통, 든든함, 중식","짬뽕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 보통, 든든함, 중식"
3,"붕어 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","붕어 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식"
4,"닭찜 [-] — 국물약간, 제공온도 뜨거움, 조리법 조림, 기름짐 보통, 든든함, 한식","닭찜 [-] — 국물약간, 제공온도 뜨거움, 조리법 조림, 기름짐 보통, 든든함, 한식"
5,"소고기 떡찜 [-] — 국물약간, 제공온도 뜨거움, 조리법 조림, 기름짐 보통, 든든함 보통, 한식","소고기 떡찜 [-] — 국물약간, 제공온도 뜨거움, 조리법 조림, 기름짐 보통, 든든함 보통, 한식"



### 느끼하지 않은 담백한 음식  |  필수 기름짐=낮음/보통 (느끼하지 않은) | 선호 기름짐=낮음 (담백한) | 선호 매운맛=없음/약함 (담백한)


,임베딩만,조건+재랭킹+중복제어
1,"화양적 [-] — 매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 부침, 기름짐 보통, 한식, 안주","무 된장국 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움, 한식"
2,"붕어 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","홍합 무국 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 한식"
3,"오리백숙 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 보통, 든든함, 한식","백합죽 [-] — 매운맛 없음, 국물약간, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움, 한식"
4,"명태 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","무국물 [-] — 매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 한식"
5,"꽃게 매운탕 [-] — 매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 보통, 한식","채소죽 [-] — 매운맛 없음, 국물약간, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움, 한식"


In [26]:
changed = [q for q in VALID_QUERIES
           if [it["라벨링단위ID"] for it in results_b[("임베딩만", q)]["추천"]]
           != [it["라벨링단위ID"] for it in results_b[("조건+재랭킹+중복제어", q)]["추천"]]]
print(f"Top-5 구성이 달라진 질의: {len(changed)}/{len(VALID_QUERIES)}건")
unchanged = [q for q in VALID_QUERIES if q not in changed]
print("달라지지 않은 질의:", unchanged)

Top-5 구성이 달라진 질의: 17/19건
달라지지 않은 질의: ['단짠단짠한 음식', '너무 맵지 않은 국물 요리']


## 13. 결과 저장

`run_config.json`에 임베딩 manifest, 원본 해시, 설정을 남겨 추적할 수 있게 한다.

In [27]:
run_config = {
    "실행시각": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "환경": env,
    "모델": DEFAULT_SPEC.as_dict(),
    "임베딩": {"A": ref_a, "B": ref_b},
    "원본해시": sources,
    "모드설정": {name: asdict(cfg) for name, cfg in MODES.items()},
    "채택설정": asdict(FULL),
    "질의": [{"질의": q, "유형": QUERY_KIND[q]} for q in ALL_QUERIES],
    "지표주의": "조건 준수 지표는 저장된 모델 추정 라벨 기준, 정답 데이터 없음",
    "실험": {
        "다양성": exp_diversity.reset_index().to_dict("records"),
        "가중치": exp_weights.reset_index().to_dict("records"),
        "후보수": exp_k.reset_index().to_dict("records"),
    },
}

pd.concat([rows_b, rows_a], ignore_index=True).to_csv(OUT_DIR / "recommendations.csv", index=False, encoding="utf-8-sig")
pd.concat([metrics_b, metrics_a], ignore_index=True).to_csv(OUT_DIR / "metrics.csv", index=False, encoding="utf-8-sig")
parsed_df.to_csv(OUT_DIR / "parsed_queries.csv", index=False, encoding="utf-8-sig")
with open(OUT_DIR / "parsed_queries.json", "w", encoding="utf-8") as f:
    json.dump({q: parse_query(q).to_dict() for q in ALL_QUERIES}, f, ensure_ascii=False, indent=2)
with open(OUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2, default=str)
with open(OUT_DIR / "results_B_full.json", "w", encoding="utf-8") as f:
    json.dump({f"{mode}|{q}": r for (mode, q), r in results_b.items()}, f, ensure_ascii=False, indent=1, default=str)

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:28s} {p.stat().st_size:>10,} bytes")

metrics.csv                      10,930 bytes
parsed_queries.csv                2,663 bytes
parsed_queries.json              13,763 bytes
recommendations.csv             177,089 bytes
results_B_full.json             414,027 bytes
run_config.json                  14,077 bytes


## 14. 요약과 한계

관찰 (저장된 추정 라벨 기준)
- 필수 조건 필터로 조건위반이 0이 되고, 선호 재랭킹과 후보 확장으로 선호 불일치가 줄어든다.
- 중복 제거와 메뉴군 상한 2로 같은 메뉴 반복이 줄어든다. 더 강한 다양성 제어는 정답 기준으로 손해라 후보로만 둔다.
- 메뉴 제외는 효과가 분명하고, 메뉴 가점 0.15는 손해가 없어 유지하되 근거가 약하다.

한계
- 라벨이 모델 추정이고 든든함은 63%가 미확인이다.
- 파서는 표에 있는 표현만 지원한다. 스키마에 없는 맛(상큼, 단맛)은 임베딩에만 맡긴다.
- 공공 데이터로 한정해 피자, 샌드위치, 치킨 계열은 후보가 거의 없다.

6단계에서 정답 세트로 설정을 비교한다.